# DITSB-v2 连续流模型：开源参数“热启动” (Warm-Start) 训练脚本

使用前请确保您已在顶部菜单栏 **代码执行程序 (Runtime)** -> **更改代码执行程序类型 (Change runtime type)** 中选择了 **GPU (T4/V100/A100)**。

## 1. 挂载 Google Drive
挂载网盘以便持久化保存训练好的模型权重 (Checkpoints)，防止 Colab 断线导致数据丢失。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 下载 DITSB 源代码并安装依赖
从 GitHub 拉取代码储备并安装所需的所有 Python 库。

In [ ]:
import os
import shutil
if os.path.exists("/content/DITSB"):
    os.chdir("/content")
    shutil.rmtree("/content/DITSB")

!git clone https://github.com/serh1m/DITSB.git /content/DITSB

import sys
os.chdir("/content/DITSB")
if "/content/DITSB" not in sys.path:
    sys.path.append("/content/DITSB")
print("Current working directory:", os.getcwd())

# Install dependencies
!pip install datasets transformers accelerate pyyaml wandb tqdm bitsandbytes
if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt

## 3. 登录 HuggingFace (下载开源模型所需)
要下载如 LLaMA-3 等受保护的开源基座模型进行热启动，需输入您的 Hugging Face Access Token。

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. 准备训练数据
数据将下载到 Colab 本地 `/content/dataset` 目录 (高速 SSD)。

In [ ]:
# 执行数据集准备脚本 
!python llm_training/prepare_data.py --dataset wikitext --output_dir=/content/dataset --seq_len=2048 \n

## 5. 自动修改配置
将 `config_7b.yaml` 里的数据路径指向本地目录，并将 Checkpoint 指向网盘。

In [ ]:
import yaml

config_path = 'llm_training/config_7b.yaml'

try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    config['data']['dataset_dir'] = '/content/dataset'
    config['training']['checkpoint_dir'] = '/content/drive/MyDrive/DITSB_checkpoints'
    config['training']['batch_size'] = 2  
    
    with open(config_path, 'w') as f:
        yaml.dump(config, f)
        
    print("✅ 配置文件已成功更新")
except FileNotFoundError:
    print(f"⚠️ 找不到配置文件: {config_path}")

## 6. 🔥 启动热启动训练 (Warm-Start Training)
在此处指定基础开源模型的路径 (如 `meta-llama/Meta-Llama-3-8B`)，脚本会自动将 Attention 和 FFN 参数映射到 DITSB 维度中，大幅度缩短概率流训练前期的收敛时间。

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/DITSB_checkpoints', exist_ok=True)

# 填入你想热启动的开源模型名称，如果在本地已经有了可以写本地路径
HF_MODEL_NAME = "NousResearch/Meta-Llama-3-8B"

# 启动附带 --warm_start_path 的训练脚本
!python llm_training/train_llm.py \
    --config llm_training/config_7b.yaml \
    --warm_start_path $HF_MODEL_NAME